# Segmentasi Transaksi & Deteksi Anomali Sederhana

Proyek belajar dengan data transaksi bank **sintetis**: kelompokkan perilaku nasabah dengan **K-Means**, tandai transaksi yang tidak biasa dengan **Isolation Forest**, dan tandai upaya login berulang dengan **aturan sederhana**.

**Sumber data:** dataset latihan Dicoding (adaptasi Kaggle *Bank Transaction Dataset for Fraud Detection*). Data sintetis kota-kota AS dalam USD, **tanpa label fraud asli**.

**Cara pakai:** jalankan dari atas ke bawah (*Run All*). Sel terakhir menyimpan semua file yang dibaca `app.py`.

---
# 1. Import Library & Muat Data

In [ ]:
import os
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

print("Library siap.")

In [ ]:
local_candidates = ['data/raw.xlsx', 'raw.xlsx']
local_path = next((p for p in local_candidates if os.path.exists(p)), None)
backup_url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vTbg5WVW6W3c8SPNUGc3A3AL-AG32TPEQGpdzARfNICMsLFI0LQj0jporhsLCeVhkN5AoRsTkn08AYl/pub?output=csv'

if local_path:
    df_raw = pd.read_excel(local_path)
    print(f"Data dimuat dari: {local_path}")
else:
    df_raw = pd.read_csv(backup_url)
    print("Data dimuat dari URL cadangan.")

print(f"Ukuran data: {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom")
df_raw.head()

---
# 2. Pembersihan Data
1. Buang baris duplikat.
2. Buang kolom yang tidak dipakai: ID, IP address, dan kolom tanggal.
3. Isi nilai kosong: **median** untuk angka, **modus** untuk teks (data kosong kecil, sekitar 1% per kolom).

In [ ]:
df = df_raw.copy()

print(f"Baris duplikat: {df.duplicated().sum()}")
df = df.drop_duplicates().reset_index(drop=True)

df = df.drop(columns=[
    'TransactionID', 'AccountID', 'DeviceID', 'MerchantID',
    'IP Address', 'PreviousTransactionDate', 'TransactionDate',
])

print("\nData kosong per kolom (sebelum diisi):")
print(df.isnull().sum()[df.isnull().sum() > 0])

for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(include=['object', 'string']).columns:
    df[col] = df[col].fillna(df[col].mode()[0])

print(f"\nUkuran data bersih: {df.shape[0]} baris, {df.shape[1]} kolom")
print(f"Sisa data kosong: {df.isnull().sum().sum()}")

---
# 3. Fitur Tambahan & Eksplorasi Singkat
- **`Balance_Ratio`**: `TransactionAmount / AccountBalance`, seberapa besar porsi saldo yang dipakai.
- **`Amount_Per_Second`**: `TransactionAmount / TransactionDuration`, kecepatan nominal transaksi.
- **`High_Login_Risk`**: aturan `LoginAttempts >= 3`. Ini **aturan**, bukan hasil model.

In [ ]:
df['Balance_Ratio'] = df['TransactionAmount'] / df['AccountBalance']
df['Amount_Per_Second'] = df['TransactionAmount'] / df['TransactionDuration'].clip(lower=1)
df['High_Login_Risk'] = df['LoginAttempts'] >= 3

numeric_cols = ['TransactionAmount', 'CustomerAge', 'TransactionDuration',
                'LoginAttempts', 'AccountBalance', 'Balance_Ratio', 'Amount_Per_Second']

print(f"Transaksi dengan LoginAttempts >= 3: {int(df['High_Login_Risk'].sum())} ({df['High_Login_Risk'].mean() * 100:.2f}%)\n")
display(df[numeric_cols].describe().T)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), numeric_cols):
    sns.histplot(df[col], bins=25, color='steelblue', ax=ax)
    ax.set_title(col, fontweight='bold')
axes.flatten()[-1].axis('off')
plt.suptitle('Sebaran Fitur Numerik', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 4. Segmentasi dengan K-Means
Aturan yang dipakai:
- **5 fitur perilaku saja**: `TransactionAmount`, `CustomerAge`, `TransactionDuration`, `AccountBalance`, `Balance_Ratio`. `LoginAttempts` sengaja tidak dipakai, supaya cluster tidak sekadar meniru aturan login.
- **`StandardScaler`** dulu, supaya `AccountBalance` (ribuan USD) tidak mendominasi fitur lain.
- **Jumlah cluster** dipilih dengan Elbow dan Silhouette Score untuk `k = 2` sampai `8`.

In [ ]:
clustering_features = ['TransactionAmount', 'CustomerAge', 'TransactionDuration',
                       'AccountBalance', 'Balance_Ratio']

scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df[clustering_features]), columns=clustering_features)

rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(df_scaled)
    rows.append({'k': k, 'Inertia': km.inertia_, 'Silhouette Score': silhouette_score(df_scaled, labels)})

eval_df = pd.DataFrame(rows)
display(eval_df)

best_k = int(eval_df.loc[eval_df['Silhouette Score'].idxmax(), 'k'])
print(f"k terbaik menurut Silhouette Score: {best_k}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(eval_df['k'], eval_df['Inertia'], marker='o')
axes[0].set_title('Elbow (Inertia)')
axes[1].plot(eval_df['k'], eval_df['Silhouette Score'], marker='s', color='seagreen')
axes[1].set_title('Silhouette Score')
for ax in axes:
    ax.set_xlabel('k')
plt.tight_layout()
plt.show()

In [ ]:
optimal_k = 2   # Silhouette tertinggi pada k = 2 (nilainya masih rendah, lihat catatan di bawah)

kmeans_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['Cluster'] = kmeans_model.fit_predict(df_scaled)

# Beri nama segmen dari rata-rata usia, supaya tidak bergantung pada nomor cluster
age_by_cluster = df.groupby('Cluster')['CustomerAge'].mean().sort_values()
name_map = {
    age_by_cluster.index[0]: 'Nasabah Muda Aktif',
    age_by_cluster.index[1]: 'Nasabah Mapan Konservatif',
}
df['Segment_Name'] = df['Cluster'].map(name_map)

print("=== Jumlah per segmen ===")
display(df['Segment_Name'].value_counts().to_frame('Jumlah').assign(**{'Persen (%)': lambda d: d['Jumlah'] / len(df) * 100}))

print("\n=== Rata-rata fitur per segmen ===")
display(df.groupby('Segment_Name')[clustering_features + ['LoginAttempts']].mean())

print("\n=== Segmen vs aturan risiko login ===")
display(pd.crosstab(df['Segment_Name'], df['High_Login_Risk']))

**Catatan:** Silhouette Score di bawah 0.5 berarti dua segmen ini saling tumpang tindih. Pada dasarnya K-Means hanya memisahkan **nasabah muda bersaldo kecil** dan **nasabah lebih tua bersaldo besar**. Berguna sebagai gambaran umum, bukan pemisahan yang tegas.

---
# 5. Deteksi Anomali dengan Isolation Forest
Satu model saja, supaya mudah dijelaskan: **Isolation Forest** mengisolasi transaksi yang "jauh" dari kebanyakan transaksi lain.

- **Fitur**: 5 fitur perilaku + `Amount_Per_Second`. `LoginAttempts` tidak ikut, supaya hasil model dan aturan login tidak saling menumpuk.
- **`contamination = 0.05`** artinya model diminta menandai sekitar 5% transaksi. Ini **pilihan saya**, bukan temuan dari data.
- Hasilnya adalah **kandidat untuk ditinjau**, bukan bukti fraud. Data ini tidak punya label fraud asli.

In [ ]:
CONTAMINATION = 0.05
anomaly_features = clustering_features + ['Amount_Per_Second']

scaler_anom = StandardScaler()
X_anom = scaler_anom.fit_transform(df[anomaly_features])

iso = IsolationForest(n_estimators=200, contamination=CONTAMINATION, random_state=42)
iso.fit(X_anom)

df['Anomaly_Flag'] = iso.predict(X_anom) == -1

# Skor 0-100: makin tinggi makin tidak biasa
iso_raw = -iso.decision_function(X_anom)
score_bounds = (float(iso_raw.min()), float(iso_raw.max()))
df['Anomaly_Score'] = (iso_raw - score_bounds[0]) / (score_bounds[1] - score_bounds[0]) * 100

print(f"Transaksi ditandai anomali: {int(df['Anomaly_Flag'].sum())} ({df['Anomaly_Flag'].mean() * 100:.2f}%)\n")
print("Rata-rata fitur: anomali vs bukan anomali")
display(df.groupby('Anomaly_Flag')[anomaly_features + ['LoginAttempts']].mean().round(2))

print("\n10 transaksi dengan skor tertinggi:")
display(df.sort_values('Anomaly_Score', ascending=False)[
    ['TransactionAmount', 'AccountBalance', 'Balance_Ratio', 'TransactionDuration', 'Anomaly_Score']
].head(10))

## 5.1 Seberapa Bergantung Hasil pada `contamination`?
Coba `0.03`, `0.05`, dan `0.07`. Kemiripan dengan daftar `0.05` diukur dengan *Jaccard index* (1.0 = identik).

In [ ]:
base_flags = df['Anomaly_Flag'].to_numpy()
rows = []
for c in [0.03, 0.05, 0.07]:
    m = IsolationForest(n_estimators=200, contamination=c, random_state=42).fit(X_anom)
    flags = m.predict(X_anom) == -1
    union = (flags | base_flags).sum()
    rows.append({
        'contamination': c,
        'Jumlah anomali': int(flags.sum()),
        'Jaccard vs 0.05': round((flags & base_flags).sum() / union, 3),
    })
display(pd.DataFrame(rows))

print("\nAnomali vs risiko login:")
display(pd.crosstab(df['High_Login_Risk'], df['Anomaly_Flag']))

---
# 6. Simpan Hasil untuk Dashboard
File yang dibaca `app.py`:
- `data/data_clustering_clean.csv`: data lengkap dengan segmen, flag anomali, dan skor
- `models/scaler_clustering.joblib`, `models/kmeans_model.joblib`
- `models/anomaly_model.joblib`: scaler + Isolation Forest + batas skor
- `models/k_evaluation.csv`: hasil Elbow/Silhouette

In [ ]:
def out_path(folder, name):
    return f"{folder}/{name}" if os.path.isdir(folder) else name

anomaly_bundle = {
    'features': anomaly_features,
    'scaler': scaler_anom,
    'model': iso,
    'score_bounds': score_bounds,
    'contamination': CONTAMINATION,
}

df.to_csv(out_path('data', 'data_clustering_clean.csv'), index=False)
joblib.dump(scaler, out_path('models', 'scaler_clustering.joblib'))
joblib.dump(kmeans_model, out_path('models', 'kmeans_model.joblib'))
joblib.dump(anomaly_bundle, out_path('models', 'anomaly_model.joblib'))
eval_df.to_csv(out_path('models', 'k_evaluation.csv'), index=False)

print("File tersimpan.\n")
print("=== Ringkasan untuk README ===")
print(f"Total transaksi          : {len(df):,}")
print(f"Silhouette (k = 2)       : {eval_df.loc[eval_df['k'] == 2, 'Silhouette Score'].iloc[0]:.4f}")
for name, n in df['Segment_Name'].value_counts().items():
    print(f"Segmen {name:<26}: {n:,} ({n / len(df) * 100:.2f}%)")
print(f"Anomali (Isolation Forest): {int(df['Anomaly_Flag'].sum())} ({df['Anomaly_Flag'].mean() * 100:.2f}%)")
print(f"Risiko login (>= 3)      : {int(df['High_Login_Risk'].sum())} ({df['High_Login_Risk'].mean() * 100:.2f}%)")